In [7]:
from random import randint
from typing import TypedDict, Literal, Sequence

from anthropic.types.beta import messages
from langchain_core.messages import ToolMessage, SystemMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from langchain.tools import tool
from timm.models.vovnet import ese_vovnet19b_slim

load_dotenv(override=True)
from rich import print

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


# 使用代码函数模拟代替外部API调用的天气查询
@tool(parse_docstring=True)
def get_weather(city: str = "上海"):
    """
    查询指定城市当日天气

    Args:
        city: 城市名称
    """
    return f"{city}的天气是晴朗的"


@tool(parse_docstring=True)
def get_news(domain: Literal["AI", "食品安全"]):
    """
    查询特定领域的当日热点

    Args:
        domain: 特定领域
    """
    if domain == "AI":
        return "Anthropic 发布了 Claude Opus-4.8，但通过 API 用中文向它发送“你是谁？”时，大多数情况下返回的却是“Qwen”或“Deepseek”。"
    elif domain == "食品安全":
        return "双汇发展子公司猪肉产品被抽检出抗生素超标37.5倍"
    else:
        return "未知的新闻领域"


# 绑定工具到模型
tools = [get_weather, get_news]
model_with_tool = model.bind_tools(tools=tools)


#1. 声明状态
class OverAllState(MessagesState):
    user_input:str
    final_output:str

#2. 声明节点
#2.1 输入节点 => 将用户输入的查询信息  记录到message中 方便后续的大模型调用
def input_node(state:OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage(state["user_input"])]
    }

#2.2 大模型节点
def llm_node(state:OverAllState) -> Command[Literal["tool_node","output_node"]]:
    # 使用goto方式完成动态调用循环结构
    ai_msg = model_with_tool.invoke(state["messages"])
    #判断接下来需要调用的节点
    if ai_msg.tool_calls:
        goto = "tool_node"
    else:
        goto = "output_node"
    return Command(
        update={
            "messages" : [ai_msg]
        },
        goto=goto
    )


#2.3 判断是否需要调用工具
def tool_node(state:OverAllState) -> OverAllState:
    # 判断大模型的返回信息 是否需要调用tool
    messages = state["messages"]
    ai_msg = messages[-1]
    tool_calls = ai_msg.tool_calls

    # 6表示函数调用60%失败
    fail_prob = 6

    for tool_call in tool_calls:
        if tool_call["name"]== "get_weather":
            # 生成[0-9]数字 判断大小
            if randint(0,9) < fail_prob:
                messages.append(ToolMessage(
                    content="网络波动,调用失败,请重试",
                    tool_call_id = tool_call["id"]
                ))
            else:
                messages.append(get_weather.invoke(tool_call))
        elif tool_call["name"] == "get_news":
            if randint(0,9) < fail_prob:
                messages.append(ToolMessage(
                    content="网络波动,调用失败,请重试",
                    tool_call_id = tool_call["id"]
                ))
            else:
                messages.append(get_news.invoke(tool_call))
        else:
            messages.append(ToolMessage(
                content="工具名称错误,调用失败,请重试",
                 tool_call_id = tool_call["id"]
                 ))
    return {
        "messages":messages
    }


#2.4 返回输出节点
def output_node(state:OverAllState) -> OverAllState:
    return {
        "final_output":state["messages"][-1].content
    }



#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("input_node",input_node)
builder.add_node("llm_node",llm_node)
builder.add_node("tool_node",tool_node)
builder.add_node("output_node",output_node)

builder.add_edge(START,"input_node")
builder.add_edge("input_node","llm_node")
builder.add_edge("tool_node","llm_node")
builder.add_edge("output_node",END)

graph = builder.compile()

ai_res = graph.invoke({
    "user_input":"查询今天的上海天气和AI新闻热点",
    "messages":[SystemMessage("如果工具调用失败,必须重新调用直到成功为止")]
})

print("user_input:",ai_res["user_input"])
print("final_output:",ai_res["final_output"])
for msg in ai_res["messages"]:
    msg.pretty_print()







user_input: 查询今天的上海天气和AI新闻热点

final_output: 两个查询都已成功获取。以下是查询结果：

---

### 🌤️ 上海天气
今天的上海天气为**晴朗**，适合外出活动。

---

### 🤖 AI新闻热点
**Anthropic 发布了 Claude Opus-4.8**，但通过 API 
用中文向它发送"你是谁？"时，大多数情况下返回的却是"Qwen"或"Deepseek"。

---

以上就是今天的上海天气和AI领域的热点新闻。如果您还需要其他信息，随时告诉我！

================================ System Message ================================

如果工具调用失败,必须重新调用直到成功为止
================================ Human Message =================================

查询今天的上海天气和AI新闻热点
================================== Ai Message ==================================

我来帮您查询今天的上海天气和AI新闻热点。
Tool Calls:
  get_weather (call_00_sgJ6ohGn04ywjjR5YVBZ2200)
 Call ID: call_00_sgJ6ohGn04ywjjR5YVBZ2200
  Args:
    city: 上海
  get_news (call_01_uloJwDTD2b8PL3JRncTb0178)
 Call ID: call_01_uloJwDTD2b8PL3JRncTb0178
  Args:
    domain: AI
================================= Tool Message =================================

网络波动,调用失败,请重试
================================= Tool Message =================================

网络波动,调用失败,请重试
================================== Ai Message ==================================

两个查询都因网络波动失败了。根据规则，我需要重新调用直到成功为止。让我重试这两个查询。
Tool Calls:
  get_weather (call_00_mUjTM7kEJ4hRKVeFyntE2830)
 Call ID: call_00_mUjTM7kEJ4hRKVeFyntE2830
  Args:
    city: 上海
  get_news (c